# Create BM25 Index and Test Retrievers

? ???? ??? ?????.
1. ChromaDB ????? BM25 ??? ??
2. BM25Retriever ?? ???
3. HybridFusion(Dense + BM25) ?? ???


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: D:\AI\projects\Medical-Chatbot
SRC_DIR: D:\AI\projects\Medical-Chatbot\src


In [ ]:
from retrieval.sparse.bm25_retriever import BM25Retriever
from retrieval.hybrid.fusion import HybridFusion

COLLECTION_NAME = "medical_knowledge"
DB_PATH = str(PROJECT_ROOT / "chroma_db")
QUERY = "급성호흡곤란증후군(ARDS)의 정의와 주요 병리 기전을 설명하시오."
TOP_K = 5

print("COLLECTION_NAME:", COLLECTION_NAME)
print("DB_PATH:", DB_PATH)
print("QUERY:", QUERY)

d:\AI\projects\Medical-Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


COLLECTION_NAME: medical_knowledge
DB_PATH: D:\AI\projects\Medical-Chatbot\chroma_db
QUERY: ??? ??


In [ ]:
# 1) BM25 retriever 인덱스 생성
bm25 = BM25Retriever(collection_name=COLLECTION_NAME, db_path=DB_PATH)
bm25.rebuild_index()
print("BM25 index build done")

INFO:medical_chatbot:[BM25Retriever] Kiwi 형태소 분석기 로드 완료


In [ ]:
# 2) BM25 retriever ???
bm25_results = bm25.retrieve(QUERY, top_k=TOP_K)
print(f"BM25 results: {len(bm25_results)}")
for i, r in enumerate(bm25_results, 1):
    meta = r.get("metadata", {}) or {}
    doc_id = r.get("id")
    score = float(r.get("bm25_score", 0.0))
    source = meta.get("source_spec")
    print(f"[{i}] id={doc_id}, score={score:.4f}, source={source}")
    print(r.get("text", "")[:200].replace("\n", " "))
    print("-" * 80)


In [ ]:
# 3) Hybrid retriever 테스트
hybrid = HybridFusion(alpha=0.7, fusion_method="weighted_sum")
hybrid_results = hybrid.retrieve(QUERY, top_k=TOP_K)

print(f"Hybrid results: {len(hybrid_results)}")
for i, c in enumerate(hybrid_results, 1):
    print(f"[{i}] c_id={c.c_id}, fused_score={c.similarity_score:.4f}, source={c.source_spec}")
    print((c.content_snippet or "")[:200].replace("\n", " "))
    print("-" * 80)
